In [6]:
# =============================================================================
# NOTEBOOK: notebook_01_chembl_qsar_dataset.ipynb
# PURPOSE:  Build a curated, publication-quality DHFR inhibitor dataset
#           for QSAR model training targeting PfDHFR (PDB: 1J3I)
#
# STRATEGY: Eukaryotic + apicomplexan DHFR targets only (see Methods rationale)
# OUTPUT:   data/chembl_dhfr_curated.parquet
#           data/chembl_dhfr_train_test_external.parquet
#
# AUTHOR:   Kenneth Odoh Chidiebere
# DATE:     April 2026
# =============================================================================

import pandas as pd
import numpy as np
import time
import os
from chembl_webresource_client.new_client import new_client

activity = new_client.activity

OUTPUT_DIR = r"C:\my_projects_all\portfolio_projects\Msc_project\data"

# ── Selected targets — eukaryotic/apicomplexan DHFR only ─────────────────────
# Bacterial and fungal DHFR excluded due to structural divergence from PfDHFR
SELECTED_TARGETS = {
    "CHEMBL1939"   : "PfDHFR-TS K1 (P. falciparum, resistant)",
    "CHEMBL4296323": "PfDHFR-TS 3D7 (P. falciparum, wild-type)",
    "CHEMBL3963"   : "PbDHFR-TS (P. berghei, rodent malaria)",
    "CHEMBL2425"   : "TgDHFR-TS (T. gondii, apicomplexan)",
    "CHEMBL202"    : "HsDHFR (H. sapiens)",
    "CHEMBL2363"   : "RnDHFR (R. norvegicus)",
    "CHEMBL4564"   : "MmDHFR (M. musculus)",
    "CHEMBL2575"   : "GgDHFR (G. gallus)",
}

def fetch_target_with_retry(target_id, label, max_retries=3, delay=10):
    """
    Fetch IC50 and Ki records for a single target with retry logic.
    Retries on connection errors with exponential backoff.
    Returns a list of activity record dicts, or empty list on failure.
    """
    for attempt in range(1, max_retries + 1):
        try:
            records = []

            for activity_type in ["IC50", "Ki"]:
                acts = activity.filter(
                    target_chembl_id=target_id,
                    standard_type=activity_type,
                    standard_relation="=",
                    standard_value__isnull=False
                ).only([
                    "molecule_chembl_id", "canonical_smiles",
                    "standard_type", "standard_value", "standard_units",
                    "assay_chembl_id", "target_chembl_id"
                ])
                batch = list(acts)
                for r in batch:
                    r["target_label"] = label
                records.extend(batch)

            return records

        except Exception as e:
            print(f"    Attempt {attempt}/{max_retries} failed: {type(e).__name__}")
            if attempt < max_retries:
                wait = delay * attempt
                print(f"    Waiting {wait}s before retry...")
                time.sleep(wait)
            else:
                print(f"    All retries exhausted for {target_id}. Skipping.")
                return []

In [7]:
# ── Fetch all selected targets ────────────────────────────────────────────────
print("Fetching ChEMBL activity records for selected DHFR targets...")
print("Retry logic active — will recover from connection drops.\n")

all_records = []

for target_id, label in SELECTED_TARGETS.items():
    print(f"  Fetching {target_id} — {label}...")
    records = fetch_target_with_retry(target_id, label)
    unique  = len(set(r["molecule_chembl_id"] for r in records))
    print(f"    → {len(records):,} records | {unique:,} unique compounds")
    all_records.extend(records)
    time.sleep(3)  # Polite pause between targets — prevents server disconnection

df_raw = pd.DataFrame(all_records)

print(f"\nRaw combined dataset:")
print(f"  Total records    : {len(df_raw):,}")
print(f"  Unique compounds : {df_raw['molecule_chembl_id'].nunique():,}")
print(f"  Missing SMILES   : {df_raw['canonical_smiles'].isna().sum():,}")
print(f"\nUnits distribution:")
print(df_raw["standard_units"].value_counts())

# Save raw — always preserve the audit trail before any curation
df_raw.to_csv(os.path.join(OUTPUT_DIR, "chembl_dhfr_raw.csv"), index=False)
print(f"\nRaw data saved to chembl_dhfr_raw.csv")

Fetching ChEMBL activity records for selected DHFR targets...
Retry logic active — will recover from connection drops.

  Fetching CHEMBL1939 — PfDHFR-TS K1 (P. falciparum, resistant)...
    Attempt 1/3 failed: ConnectionError
    Waiting 10s before retry...
    → 1,077 records | 193 unique compounds
  Fetching CHEMBL4296323 — PfDHFR-TS 3D7 (P. falciparum, wild-type)...
    → 4 records | 2 unique compounds
  Fetching CHEMBL3963 — PbDHFR-TS (P. berghei, rodent malaria)...
    → 42 records | 35 unique compounds
  Fetching CHEMBL2425 — TgDHFR-TS (T. gondii, apicomplexan)...
    Attempt 1/3 failed: ConnectionError
    Waiting 10s before retry...
    Attempt 2/3 failed: ConnectionError
    Waiting 20s before retry...
    → 1,489 records | 720 unique compounds
  Fetching CHEMBL202 — HsDHFR (H. sapiens)...
    → 1,787 records | 1,257 unique compounds
  Fetching CHEMBL2363 — RnDHFR (R. norvegicus)...
    → 1,731 records | 1,009 unique compounds
  Fetching CHEMBL4564 — MmDHFR (M. musculus)...
 

In [8]:
# =============================================================================
# SECTION 2: CURATION — STANDARDIZE UNITS AND CALCULATE pIC50
#
# Critical step: mixed units (nM, uM, mM) must be converted to nM before
# pIC50 calculation. Unit errors here corrupt the entire QSAR dataset.
# This is the most common methodological flaw in published QSAR papers.
#
# pIC50 = -log10(IC50 in Molar)
#       = -log10(IC50_nM * 1e-9)
#       =  9 - log10(IC50_nM)
#
# Activity threshold: pIC50 >= 6.0 (IC50 <= 1000 nM) = active
#                     pIC50 <  6.0                    = inactive
# =============================================================================

df = df_raw.copy()

# ── Step 1: Remove records without SMILES ─────────────────────────────────────
df = df[df["canonical_smiles"].notna()].copy()
print(f"After removing missing SMILES: {len(df):,} records")

# ── Step 2: Convert all values to numeric ─────────────────────────────────────
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df[df["standard_value"].notna() & (df["standard_value"] > 0)].copy()
print(f"After removing non-numeric/zero values: {len(df):,} records")

# ── Step 3: Standardize units to nM ──────────────────────────────────────────
unit_map = {
    "nM" : 1.0,
    "uM" : 1e3,
    "mM" : 1e6,
    "M"  : 1e9,
    "pM" : 1e-3,
    "fM" : 1e-6,
}

# Flag records with unrecognized units before dropping
unrecognized_units = df[~df["standard_units"].isin(unit_map.keys())]["standard_units"].value_counts()
if len(unrecognized_units) > 0:
    print(f"\nUnrecognized units (will be excluded):")
    print(unrecognized_units)

df = df[df["standard_units"].isin(unit_map.keys())].copy()
df["value_nM"] = df["standard_value"] * df["standard_units"].map(unit_map)
print(f"After unit standardization: {len(df):,} records")

# ── Step 4: Calculate pIC50 ───────────────────────────────────────────────────
# pIC50 = 9 - log10(value_nM)  [equivalent to -log10(value_M)]
df["pIC50"] = 9 - np.log10(df["value_nM"])

# Sanity check — remove physically unreasonable values
# pIC50 < 3 (IC50 > 1 mM) or pIC50 > 12 (IC50 < 1 pM) indicate data errors
n_before = len(df)
df = df[(df["pIC50"] >= 3.0) & (df["pIC50"] <= 12.0)].copy()
print(f"After pIC50 range filter (3–12): {len(df):,} records "
      f"(removed {n_before - len(df)} outliers)")

print(f"\npIC50 distribution:")
print(df["pIC50"].describe().round(3))

After removing missing SMILES: 6,600 records
After removing non-numeric/zero values: 6,600 records

Unrecognized units (will be excluded):
standard_units
kcat/Km    8
ug.mL-1    7
Name: count, dtype: int64
After unit standardization: 6,585 records
After pIC50 range filter (3–12): 6,511 records (removed 74 outliers)

pIC50 distribution:
count    6511.000
mean        6.470
std         1.523
min         3.009
25%         5.391
50%         6.432
75%         7.538
max        11.921
Name: pIC50, dtype: float64


In [9]:
# =============================================================================
# SECTION 3: DEDUPLICATION — ONE RECORD PER COMPOUND
#
# Multiple assays report activity for the same compound against the same target.
# Strategy: take the median pIC50 across all records for each unique SMILES.
# Median is preferred over mean — it is robust to outlier assay values.
#
# Deduplication key: canonical_smiles (not molecule_chembl_id, because the
# same structure can have multiple ChEMBL IDs due to salt/stereoisomer variants)
# =============================================================================

# ── Step 1: Standardize SMILES via RDKit canonicalization ────────────────────
from rdkit import Chem

def canonicalize(smi):
    """Return RDKit canonical SMILES, or None if invalid."""
    try:
        mol = Chem.MolFromSmiles(str(smi))
        return Chem.MolToSmiles(mol) if mol else None
    except:
        return None

print("Canonicalizing SMILES...")
df["smiles_canonical"] = df["canonical_smiles"].apply(canonicalize)
df = df[df["smiles_canonical"].notna()].copy()
print(f"After SMILES canonicalization: {len(df):,} records")

# ── Step 2: Deduplicate — median pIC50 per unique SMILES ─────────────────────
deduped = (
    df.groupby("smiles_canonical")
    .agg(
        pIC50_median      = ("pIC50", "median"),
        pIC50_std         = ("pIC50", "std"),
        n_measurements    = ("pIC50", "count"),
        molecule_chembl_id= ("molecule_chembl_id", "first"),
        # Track which targets this compound was measured against
        targets_measured  = ("target_label", lambda x: "; ".join(sorted(set(x)))),
        has_pf_data       = ("target_chembl_id",
                             lambda x: any(t in ["CHEMBL1939", "CHEMBL4296323"]
                                           for t in x)),
    )
    .reset_index()
    .rename(columns={"pIC50_median": "pIC50"})
)

print(f"\nAfter deduplication:")
print(f"  Unique compounds         : {len(deduped):,}")
print(f"  With PfDHFR measurements : {deduped['has_pf_data'].sum():,}")
print(f"  Measured in >1 assay     : {(deduped['n_measurements'] > 1).sum():,}")

# ── Step 3: Assign binary activity label ─────────────────────────────────────
# pIC50 >= 6.0 = active (IC50 <= 1000 nM)
# pIC50 <  6.0 = inactive
deduped["active"] = (deduped["pIC50"] >= 6.0).astype(int)

print(f"\nActivity distribution (threshold: pIC50 >= 6.0):")
print(f"  Active   (1): {deduped['active'].sum():,} "
      f"({deduped['active'].mean()*100:.1f}%)")
print(f"  Inactive (0): {(deduped['active']==0).sum():,} "
      f"({(1-deduped['active'].mean())*100:.1f}%)")

Canonicalizing SMILES...
After SMILES canonicalization: 6,511 records

After deduplication:
  Unique compounds         : 2,537
  With PfDHFR measurements : 193
  Measured in >1 assay     : 1,108

Activity distribution (threshold: pIC50 >= 6.0):
  Active   (1): 1,400 (55.2%)
  Inactive (0): 1,137 (44.8%)


In [10]:
# =============================================================================
# SECTION 4: SCAFFOLD-BASED TRAIN / TEST / EXTERNAL SPLIT
#
# Random splits inflate performance because structurally similar compounds
# end up in both train and test sets. Scaffold-based splitting ensures that
# compounds sharing the same Murcko scaffold go entirely into one split.
#
# Split ratios: 70% train / 15% test / 15% external validation
#
# The external set is LOCKED after splitting — it is not used during model
# development and is only evaluated once at the very end.
# Compounds with PfDHFR-specific measurements are preferentially assigned
# to the external set to serve as the prospective validation benchmark.
# =============================================================================

from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def get_scaffold(smiles):
    """Extract Murcko scaffold SMILES from a compound SMILES."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except:
        return None

print("Computing Murcko scaffolds...")
deduped["scaffold"] = deduped["smiles_canonical"].apply(get_scaffold)

n_unique_scaffolds = deduped["scaffold"].nunique()
print(f"Unique Murcko scaffolds: {n_unique_scaffolds:,}")
print(f"Compounds per scaffold (median): "
      f"{deduped.groupby('scaffold').size().median():.1f}")

# ── Scaffold-based split ──────────────────────────────────────────────────────
np.random.seed(42)

# Group compounds by scaffold
scaffold_groups = defaultdict(list)
for idx, row in deduped.iterrows():
    scaffold_groups[row["scaffold"]].append(idx)

# Sort scaffolds by size (largest first) for balanced splitting
scaffolds_sorted = sorted(
    scaffold_groups.items(),
    key=lambda x: len(x[1]),
    reverse=True
)

train_idx, test_idx, external_idx = [], [], []
train_target = int(0.70 * len(deduped))
test_target  = int(0.15 * len(deduped))

for scaffold, indices in scaffolds_sorted:
    # PfDHFR compounds go to external set preferentially
    pf_in_group = deduped.loc[indices, "has_pf_data"].any()

    if pf_in_group and len(external_idx) < int(0.15 * len(deduped)):
        external_idx.extend(indices)
    elif len(train_idx) < train_target:
        train_idx.extend(indices)
    elif len(test_idx) < test_target:
        test_idx.extend(indices)
    else:
        external_idx.extend(indices)

# Assign split labels
deduped["split"] = "train"
deduped.loc[test_idx,     "split"] = "test"
deduped.loc[external_idx, "split"] = "external"

print(f"\nDataset split summary:")
print(f"  Train    : {(deduped['split']=='train').sum():,} compounds")
print(f"  Test     : {(deduped['split']=='test').sum():,} compounds")
print(f"  External : {(deduped['split']=='external').sum():,} compounds")
print(f"\nPfDHFR compounds by split:")
pf = deduped[deduped["has_pf_data"]]
print(pf["split"].value_counts())
print(f"\nActivity balance by split:")
print(deduped.groupby("split")["active"].mean().round(3))

Computing Murcko scaffolds...
Unique Murcko scaffolds: 714
Compounds per scaffold (median): 1.0

Dataset split summary:
  Train    : 1,775 compounds
  Test     : 374 compounds
  External : 388 compounds

PfDHFR compounds by split:
split
train       116
external     49
test         28
Name: count, dtype: int64

Activity balance by split:
split
external    0.497
test        0.449
train       0.585
Name: active, dtype: float64


In [11]:
# =============================================================================
# SECTION 5: SAVE CURATED DATASET
# =============================================================================

out_path = os.path.join(OUTPUT_DIR, "chembl_dhfr_curated.parquet")
deduped.to_parquet(out_path, index=False)
print(f"Saved: chembl_dhfr_curated.parquet | Shape: {deduped.shape}")

# Human-readable CSV for inspection
csv_path = os.path.join(OUTPUT_DIR, "chembl_dhfr_curated_preview.csv")
deduped.to_csv(csv_path, index=False)
print(f"Saved: chembl_dhfr_curated_preview.csv")

print(f"""
=============================================================
  CHEMBL DHFR DATASET CURATION REPORT
=============================================================
Targets included     : {len(SELECTED_TARGETS)} eukaryotic/apicomplexan DHFR
Raw records fetched  : (see chembl_dhfr_raw.csv)
Final unique compounds: {len(deduped):,}
  Active (pIC50>=6.0): {deduped['active'].sum():,}
  Inactive           : {(deduped['active']==0).sum():,}
  With PfDHFR data   : {deduped['has_pf_data'].sum():,}

Split (scaffold-based):
  Train    : {(deduped['split']=='train').sum():,}
  Test     : {(deduped['split']=='test').sum():,}
  External : {(deduped['split']=='external').sum():,}

Next: notebook_02_qsar_modeling.ipynb
=============================================================
""")

Saved: chembl_dhfr_curated.parquet | Shape: (2537, 10)
Saved: chembl_dhfr_curated_preview.csv

  CHEMBL DHFR DATASET CURATION REPORT
Targets included     : 8 eukaryotic/apicomplexan DHFR
Raw records fetched  : (see chembl_dhfr_raw.csv)
Final unique compounds: 2,537
  Active (pIC50>=6.0): 1,400
  Inactive           : 1,137
  With PfDHFR data   : 193

Split (scaffold-based):
  Train    : 1,775
  Test     : 374
  External : 388

Next: notebook_02_qsar_modeling.ipynb

